# 01 · Exploración de bases de datos

**Curso:** Análisis de Datos — ITM · **Evento evaluativo 2** · Semestre 2026-2

**Objetivo de este notebook (Fase 1):** explorar varias bases de datos de distintos tipos, documentar sus características y elegir, con criterios explícitos, la base sobre la que se hará el EDA y el preprocesamiento.

Se exploraron **4 bases de datos de 3 tipos diferentes** (el enunciado pide mínimo 3 bases de 2 tipos):

| # | Base | Tipo de dato | Fuente |
|---|------|--------------|--------|
| 1 | Titanic | Tabular | Repositorio `seaborn-data` (derivado del dataset público de Kaggle / Encyclopedia Titanica) |
| 2 | Viajes de taxi NYC (marzo 2019) | Tabular con fechas | Repositorio `seaborn-data` (muestra de los registros de la NYC Taxi & Limousine Commission) |
| 3 | SMS Spam Collection | Texto | UCI Machine Learning Repository (copia en GitHub) |
| 4 | Digits (dígitos manuscritos 8×8) | Imágenes | UCI / incluido en scikit-learn |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.datasets import load_digits

sns.set_theme(style="whitegrid", palette="deep")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/titanic.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecutar desde el repositorio o su carpeta notebooks.")
DATA = ROOT / "data"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
OUT = ROOT / "reports" / "mejoras"
OUT.mkdir(parents=True, exist_ok=True)

def tamano_mb(ruta):
    return round(Path(ruta).stat().st_size / 1024**2, 3)

## 1. Titanic (tabular)

- **Fuente:** registros de pasajeros del RMS Titanic (1912), recopilados por Encyclopedia Titanica y publicados en Kaggle; se usa la versión del repositorio `seaborn-data`.
- **Tipo según origen:** **secundaria** — los datos fueron recolectados por terceros (listas de pasajeros) y no por el equipo.
- **Posibles aplicaciones:** clasificación (predecir supervivencia), análisis de desigualdad social y de protocolos de evacuación.

In [2]:
titanic = pd.read_csv(DATA / "titanic.csv")
print("Dimensiones:", titanic.shape)
print("Tamaño en disco:", tamano_mb(DATA / "titanic.csv"), "MB")
print("\nTipos de datos:\n", titanic.dtypes.value_counts())
print("\n% de valores faltantes por columna:")
print((titanic.isna().mean()*100).round(1).sort_values(ascending=False).head(5))
titanic.head()

Dimensiones: (891, 15)
Tamaño en disco: 0.054 MB

Tipos de datos:
 str        7
int64      4
float64    2
bool       2
Name: count, dtype: int64

% de valores faltantes por columna:
deck           77.2
age            19.9
embarked        0.2
embark_town     0.2
survived        0.0
dtype: float64


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Viajes de taxi en Nueva York (tabular + fechas)

- **Fuente:** muestra de viajes de taxis amarillos y verdes de la NYC Taxi & Limousine Commission (marzo de 2019), publicada en `seaborn-data`.
- **Tipo según origen:** **secundaria** para el equipo: usamos registros de terceros en una muestra procesada. La transformación del archivo se documenta por separado del origen de la fuente.
- **Posibles aplicaciones:** predicción de tarifas y propinas, análisis de demanda por hora y zona, series de tiempo.

In [3]:
taxis = pd.read_csv(DATA / "taxis.csv", parse_dates=["pickup", "dropoff"])
print("Dimensiones:", taxis.shape)
print("Tamaño en disco:", tamano_mb(DATA / "taxis.csv"), "MB")
print("Rango de fechas:", taxis.pickup.min(), "→", taxis.pickup.max())
print("\n% de valores faltantes por columna:")
print((taxis.isna().mean()*100).round(2).sort_values(ascending=False).head(5))
taxis.head()

Dimensiones: (6433, 14)
Tamaño en disco: 0.829 MB
Rango de fechas: 2019-02-28 23:29:03 → 2019-03-31 23:43:45

% de valores faltantes por columna:
dropoff_zone       0.70
dropoff_borough    0.70
payment            0.68
pickup_zone        0.40
pickup_borough     0.40
dtype: float64


,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


## 3. SMS Spam Collection (texto)

- **Fuente:** Almeida & Gómez Hidalgo (2011), UCI Machine Learning Repository. Mensajes SMS en inglés etiquetados como `ham` (legítimo) o `spam`.
- **Tipo según origen:** **secundaria** (recolectada por investigadores a partir de foros y corpus públicos).
- **Posibles aplicaciones:** clasificación de texto, detección de spam, procesamiento de lenguaje natural (TF-IDF, embeddings).

In [4]:
sms = pd.read_csv(DATA / "sms_spam.tsv", sep="\t", header=None, names=["etiqueta", "mensaje"])
sms["longitud"] = sms["mensaje"].str.len()
print("Dimensiones:", sms.shape)
print("Tamaño en disco:", tamano_mb(DATA / "sms_spam.tsv"), "MB")
print("\nDistribución de clases:\n", sms["etiqueta"].value_counts(normalize=True).round(3))
print("\nLongitud media por clase:\n", sms.groupby("etiqueta")["longitud"].mean().round(1))
sms.head()

Dimensiones: (5572, 3)
Tamaño en disco: 0.456 MB

Distribución de clases:
 etiqueta
ham     0.866
spam    0.134
Name: proportion, dtype: float64

Longitud media por clase:
 etiqueta
ham      71.5
spam    138.7
Name: longitud, dtype: float64


,etiqueta,mensaje,longitud
0,ham,"Go until jurong point, crazy.. Available only ...",111
1,ham,Ok lar... Joking wif u oni...,29
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155
3,ham,U dun say so early hor... U c already then say...,49
4,ham,"Nah I don't think he goes to usf, he lives aro...",61


## 4. Digits — dígitos manuscritos (imágenes)

- **Fuente:** *Optical Recognition of Handwritten Digits*, UCI Machine Learning Repository; incluido en `scikit-learn`.
- **Tipo según origen:** **secundaria** — las imágenes originales (32×32) fueron reducidas a 8×8 y normalizadas por los autores.
- **Posibles aplicaciones:** clasificación de imágenes, visión por computador introductoria, ejemplos de reducción de dimensionalidad.

In [5]:
digits = load_digits()
print("Número de imágenes:", digits.images.shape[0])
print("Resolución:", digits.images.shape[1:], "→", digits.data.shape[1], "atributos (píxeles)")
print("Clases:", np.unique(digits.target))
print("Tamaño en memoria:", round(digits.data.nbytes/1024**2, 3), "MB")

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, img, lab in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(lab); ax.axis("off")
plt.suptitle("Muestra de imágenes del dataset Digits")
plt.tight_layout(); plt.savefig(FIG / "01_muestra_digits.png", dpi=120); plt.show()

Número de imágenes: 1797
Resolución: (8, 8) → 64 atributos (píxeles)
Clases: [0 1 2 3 4 5 6 7 8 9]
Tamaño en memoria: 0.877 MB


/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58023/2971245315.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "01_muestra_digits.png", dpi=120); plt.show()


## 5. Resumen comparativo

In [6]:
resumen = pd.DataFrame([
    ["Titanic", "Tabular", "Secundaria", titanic.shape[0], titanic.shape[1], tamano_mb(DATA/"titanic.csv"),
     round(titanic.isna().mean().mean()*100, 1), "Alta (diccionario conocido, muy usada)"],
    ["Taxis NYC", "Tabular + tiempo", "Secundaria", taxis.shape[0], taxis.shape[1], tamano_mb(DATA/"taxis.csv"),
     round(taxis.isna().mean().mean()*100, 1), "Media (variables autoexplicativas, sin diccionario oficial)"],
    ["SMS Spam", "Texto", "Secundaria", sms.shape[0], 2, tamano_mb(DATA/"sms_spam.tsv"),
     round(sms[["etiqueta","mensaje"]].isna().mean().mean()*100, 1), "Alta (artículo científico + UCI)"],
    ["Digits", "Imágenes", "Secundaria", digits.data.shape[0], digits.data.shape[1],
     round(digits.data.nbytes/1024**2, 3), 0.0, "Alta (UCI + scikit-learn)"],
], columns=["Base", "Tipo", "Origen", "Registros", "Atributos", "Tamaño (MB)", "% faltantes promedio", "Documentación"])
resumen

,Base,Tipo,Origen,Registros,Atributos,Tamaño (MB),% faltantes promedio,Documentación
0,Titanic,Tabular,Secundaria,891,15,0.054,6.5,"Alta (diccionario conocido, muy usada)"
1,Taxis NYC,Tabular + tiempo,Secundaria,6433,14,0.829,0.2,"Media (variables autoexplicativas, sin diccion..."
2,SMS Spam,Texto,Secundaria,5572,2,0.456,0.0,Alta (artículo científico + UCI)
3,Digits,Imágenes,Secundaria,1797,64,0.877,0.0,Alta (UCI + scikit-learn)


## 6. Criterios de selección

El equipo asignó de 1 a 5 puntos en cinco criterios, con el mismo peso: completitud, relevancia, documentación, manejabilidad y riqueza para el EDA. Las calificaciones son juicios argumentados para esta actividad, no un ranking universal de calidad. Completitud premia menos ausencias; riqueza evalúa oportunidades para practicar varias técnicas. Son criterios distintos.

| Criterio | Qué se evalúa |
|----------|---------------|
| Completitud | Proporción de datos presentes. *Ojo:* una base perfectamente completa da menos material para practicar tratamiento de faltantes. |
| Relevancia | Si permite plantear preguntas e hipótesis interesantes. |
| Documentación | Disponibilidad de descripción de variables y fuente confiable. |
| Manejabilidad | Tamaño y facilidad para procesar en un notebook. |
| Riqueza para el EDA | Mezcla de tipos de variables, faltantes reales, outliers, posibilidad de codificar y de aplicar PCA con sentido. |

In [7]:
criterios = pd.DataFrame({
    "Completitud":        [4, 5, 5, 5],
    "Relevancia":         [5, 4, 4, 3],
    "Documentación":      [5, 3, 4, 5],
    "Manejabilidad":      [5, 5, 4, 5],
    "Riqueza para EDA":   [5, 4, 2, 2],
}, index=["Titanic", "Taxis NYC", "SMS Spam", "Digits"])
criterios["Total"] = criterios.sum(axis=1)
criterios = criterios.sort_values("Total", ascending=False)

ax = criterios.drop(columns="Total").plot(kind="barh", stacked=True, figsize=(8, 3.5), colormap="viridis")
ax.set_xlabel("Puntaje acumulado (máx. 25)"); ax.set_title("Evaluación de las bases de datos candidatas")
ax.invert_yaxis(); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig(FIG / "01_criterios_seleccion.png", dpi=120); plt.show()
criterios

/var/folders/p2/xfh16h8x59n5n_k7qs00b5vw0000gp/T/ipykernel_58023/4007621802.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "01_criterios_seleccion.png", dpi=120); plt.show()


,Completitud,Relevancia,Documentación,Manejabilidad,Riqueza para EDA,Total
Titanic,4,5,5,5,5,24
Taxis NYC,5,4,3,5,4,21
Digits,5,3,5,5,2,20
SMS Spam,5,4,4,4,2,19


## 7. Justificación de las bases exploradas y selección final

Elegimos estas cuatro candidatas para comparar tres formatos y problemas distintos, utilizando fuentes secundarias para el equipo: ninguna fue recolectada por nosotros. Titanic y Taxis son versiones procesadas de datos tabulares; SMS aporta texto y Digits representa imágenes mediante píxeles. Documentamos esas transformaciones sin asumir que conviertan automáticamente una fuente secundaria en terciaria.

| Base | Por qué incluirla en la exploración | Por qué no continuar con ella en esta entrega |
|------|-----------------------------------|-----------------------------------------------|
| Titanic | Mezcla numéricas y categóricas, ausencias, valores extremos y una respuesta interpretable. Permite comparar supervivencia por sexo, clase y edad. | Es la seleccionada. |
| Taxis NYC | Permite estudiar tiempos, trayectos, tarifas y propinas en datos operativos. | Es viable, pero tiene pocas ausencias en esta muestra y aporta menos variedad para el tratamiento de faltantes que queremos demostrar. |
| SMS Spam | Permite comparar mensajes legítimos y spam, longitud y distribución de etiquetas. Aporta un formato textual distinto. | El texto requeriría vectorización para aplicar PCA; priorizamos explorar y preparar variables tabulares interpretables. |
| Digits | Permite trabajar con imágenes pequeñas, 64 atributos de píxel y visualizar reducción dimensional. | No presenta faltantes y sus atributos son píxeles; ofrece menos oportunidades para el tratamiento de ausencias y categorías. Sigue siendo una buena opción para visión y PCA. |

**Resultado de la matriz:** Titanic 24/25; Taxis 21/25; Digits 20/25; SMS 19/25. La selección favorece la adecuación al ejercicio, no la superioridad general de una base.

**Por qué Titanic:** sus 891 registros y 15 columnas son manejables y combinan variables numéricas y categóricas. Contiene 177 edades ausentes, 688 cubiertas ausentes y 2 puertos ausentes (también reflejados en su columna redundante). Las tarifas extremas son plausibles y requieren discutir su tratamiento; `survived` permite formular hipótesis exploratorias. Las ausencias no son una ventaja de calidad, pero permiten justificar estrategias diferentes dentro del ejercicio.

Siguiente paso: `02_eda.ipynb`.

## 8. Registro del entorno

Se guardan las versiones utilizadas para facilitar la reproducción de las tres fases. Los archivos CSV locales son las entradas de la ejecución; Digits viene incluido con scikit-learn.

In [8]:
import json
import platform
import importlib.metadata as metadata
entorno = {"python": platform.python_version(), "plataforma": platform.platform(),
           "paquetes": {p: metadata.version(p) for p in
                        ["pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "seaborn"]}}
(OUT / "entorno.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False), encoding="utf-8")
print("Salidas:", OUT)

Salidas: /Users/darwin/Desktop/Analisis de datos /proyecto-analisis-datos/reports/mejoras
